In [0]:
%sql 
CREATE VOLUME databricksharis.bronze.auto_vol
--- Create a raw folder inside that upload a file order_batch_1
---- create a destination location 
----in this destination create a directory of checkpoint and data

### **AUTOLOADER QUERY**

After Evolution , it'll fail as there's schema mismatch. But after running again the Write stream , it'll write the data 



#### addNewColumns or rescue for SchemaEvolutionMode

In [0]:
df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "csv")\
    .option("cloudFiles.schemaLocation", "/Volumes/databricksharis/bronze/auto_vol/destination/checkpoint/")\
      .option("cloudFiles.schemaEvolutionMode", "rescue")\
      .load("/Volumes/databricksharis/bronze/auto_vol/raw/")
    # .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    # .option("cloudFiles.maxFilesPerTrigger", 1)

### Many options for Trigger , this trigger will and it'll stop once it loads the data 

In [0]:
df.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "/Volumes/databricksharis//bronze/auto_vol/destination/checkpoint/")\
    .trigger(once=True)\
    .start("/Volumes/databricksharis//bronze/auto_vol/destination/data/")

In [0]:
df = spark.read.format("delta").load("/Volumes/databricksharis//bronze/auto_vol/destination/data/")


- sources : responsible for ideompotency 
- Offset : it assigns a query more related to query proving a offset query
- commit : this file is processed / or any failure it'll say it's not comitted

![image_1772790374373.png](./image_1772790374373.png "image_1772790374373.png")

In [0]:
display(df)

Let's now add another file orderbatch2 with same schema and it should upload only this file and write only write stream

earlier we run only WriteStream
After adding the file with extra Schema 

when we run the entire query , it grabs the latest schema from schema location, then it simply continuing the process.

In [0]:
#data is stored in Dictionary. We cannot use JSON (Strinfied of JSon Type)
from pyspark.sql.types import StructType, StringType
from pyspark.sql.functions import from_json, col

rescued_schema = (
    StructType()
    .add("discount", StringType())
    .add("payment_method", StringType())
)

# PARSE THE STRINGFIED JSON COLUMN

df = df.withColumn(
    "rescued_struct",
    from_json(
        col("_rescued_data"),
        rescued_schema
    )
)

#EXTRACT THE INDIVIDUAL FIELDS
df = (
    df.withColumn(
        "rescued_discount",
        col("rescued_struct.discount")
    )
    .withColumn(
        "rescued_payment_method",
        col("rescued_struct.payment_method")
    )
)

display(df)

#### NOW DROP THE BOTH DIRECTORIES IN DESTINATION CHECKPOINT AND DATA AND DELETE EVOLVED FILES AS WELL IN RAW FOLDER AS WE ARE GOING WITH ADD COLUMNS OPTION

Also it'll fail and for that we have to run Read as well as Write Queries. As it'll first evolve the schema then after that we can write

In [0]:
df = spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "csv")\
    .option("cloudFiles.schemaLocation", "/Volumes/databricksharis//bronze/auto_vol/destination/checkpoint/")\
      .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
      .load("/Volumes/databricksharis//bronze/auto_vol/raw/")

We have added merge schemas in True, withouout manual intervention columns are added automatically in destination

In [0]:
df.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "/Volumes/databricksharis//bronze/auto_vol/destination/checkpoint/")\
    .option("mergeSchema", True)\
    .trigger(once=True)\
    .start("/Volumes/databricksharis//bronze/auto_vol/destination/data/") 

In [0]:
df_1 = spark.read.format("delta").load("/Volumes/databricksharis//bronze/auto_vol/destination/data/")
display(df_1)

- now Add the 3rd faulty file
After that if we run the Write Stream , it'll fail and then we have to restart the read stream and write Stream

it failed as it need to fetch the updated schema  and if you are confident enough it'll not break the data , then we can use this addNewColumns else rescued column is the safest option.

This was possible with merge Schema 

In [0]:
df_1 = spark.read.format("delta").load("/Volumes/databricksharis//bronze/auto_vol/destination/data/")
display(df_1)